# Tokenizer lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/tokenizer.ipynb)

Train a byte-level BPE tokenizer from scratch and watch it learn. Companion to the [Tokenization](../03-foundations/tokenization.md) page. Runs on CPU in seconds.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

from s2a.bpe import BPETokenizer

## 1. Train on some text

Paste in any text you like. More varied text gives more useful merges.

In [ ]:
corpus = """
Tokenization turns text into tokens. Tokens are chunks of text from a fixed vocabulary.
Byte-pair encoding starts from bytes and repeatedly merges the most frequent pair.
Frequent words become single tokens, and rare words are split into smaller pieces.
""" * 10

tok = BPETokenizer().train(corpus, vocab_size=400)
print("vocabulary size:", tok.vocab_size)

## 2. Watch the first merges

The earliest merges are the most frequent pairs: usually spaces plus common letters.

In [ ]:
for (a, b), new_id in list(tok.merges.items())[:20]:
    print(f"{tok.vocab[a]!r:>12} + {tok.vocab[b]!r:<12} -> {tok.vocab[new_id]!r}")

## 3. Tokenize new text

In [ ]:
for text in ["Tokens are frequent words.", "Untokenizable gibberish: xqzv!", "naïve café 東京 🙂"]:
    ids = tok.encode(text)
    assert tok.decode(ids) == text
    print(f"{len(text.encode()):>3} bytes -> {len(ids):>3} tokens  {tok.tokens(text)}")

The `�` pieces are single bytes of a multi-byte character (like `ï` or `東`) that the tokenizer never learned to merge. They still decode back perfectly, because decoding joins the raw bytes first.

## 4. Why pre-tokenization matters

In [ ]:
raw = BPETokenizer(pretokenize=False).train(corpus, vocab_size=400)
crossing = sorted({p.decode("utf-8", "replace") for p in raw.vocab.values() if b" " in p[1:]}, key=len)
print(f"{len(crossing)} tokens span word boundaries. Shortest examples:", crossing[:8])
print(f"Longest is {len(crossing[-1])} characters: it memorized whole phrases of this repetitive text.")

## Try it

- Increase `vocab_size`. When does compression stop improving on this corpus?
- Train on text in another language and compare tokens per byte.
- Compare with a real tokenizer: `pip install tiktoken`, then `tiktoken.get_encoding("cl100k_base").encode(text)`.